# 🚀 Huấn Luyện Mô Hình YOLO11s Cho Bộ Dữ Liệu Ảnh Nhiệt (Thermal IR Person Detection)

Notebook này cung cấp quy trình chuẩn, hoàn chỉnh và tối ưu hóa để huấn luyện mô hình **YOLO11s** (Ultralytics YOLO11 - Small model) phát hiện người trong ảnh nhiệt hồng ngoại (phục vụ bài toán Tìm kiếm & Cứu nạn - Search and Rescue / UAV Thermal Surveillance).

---

### 📌 Cấu trúc & Tỷ lệ phân chia dữ liệu (3 tập Train - Val - Test):
Mỗi tập dữ liệu đều chứa 2 thư mục con chuẩn: `image` (chứa ảnh nhiệt) và `label` (chứa nhãn YOLO .txt):
- **Tập Huấn luyện (Train)**: **19,802 ảnh** (~72%) — Dùng để cập nhật gradient mô hình.
- **Tập Kiểm định (Val)**: **2,200 ảnh** (~8%) — Dùng để tính toán validation loss và mAP sau mỗi epoch (Early Stopping).
- **Tập Kiểm tra (Test)**: **5,548 ảnh** (~20%) — Đánh giá độc lập chất lượng mô hình sau khi kết thúc huấn luyện.
- **Tổng số lượng**: **27,550 ảnh** | Lớp đối tượng: `0: person`.

---

### ⚙️ Cấu hình phần cứng mục tiêu:
- **GPU**: Hỗ trợ NVIDIA GPU (GeForce RTX 4060 Laptop GPU 8GB VRAM, RTX 3060/4070, hoặc Google Colab / Kaggle T4/A100).
- **Tối ưu**: Kích hoạt **AMP (Automatic Mixed Precision FP16)** giúp tiết kiệm bộ nhớ VRAM và tăng tốc độ xử lý gấp đôi.

In [ ]:
# ==========================================================
# BƯỚC 1: KIỂM TRA MÔI TRƯỜNG, GPU VÀ CÀI ĐẶT THƯ VIỆN
# ==========================================================
import sys
import os

# Đảm bảo mã hóa UTF-8 trên Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# Tự động kiểm tra và cài đặt ultralytics nếu cần
try:
    import ultralytics
    print(f"✅ Ultralytics version: {ultralytics.__version__}")
except ImportError:
    print("⏳ Đang cài đặt thư viện ultralytics...")
    !pip install ultralytics
    import ultralytics
    print(f"✅ Ultralytics version: {ultralytics.__version__}")

import torch
import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

print("-" * 55)
print(f"🐍 Python version : {sys.version.split()[0]}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"⚡ CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"🎮 GPU Thiết bị   : {device_name}")
    print(f"💾 Tổng dung lượng VRAM: {vram_gb:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Không có GPU CUDA, mô hình sẽ chạy trên CPU (chậm hơn nhiều).")
print("-" * 55)

## 📂 Bước 2: Kiểm Tra Số Lượng Dữ Liệu Trong 3 Tập (Train - Val - Test)

Mỗi tập đều có 2 thư mục con:
```
Thermal/
├── Train/   ->  image/ (19,802 ảnh)  và  label/ (19,802 nhãn)
├── Val/     ->  image/ (2,200 ảnh)   và  label/ (2,200 nhãn)
└── Test/    ->  image/ (5,548 ảnh)   và  label/ (5,548 nhãn)
```

> [!TIP]
> Hệ thống đã thiết lập liên kết tương thích (`image` $\leftrightarrow$ `images`, `label` $\leftrightarrow$ `labels`) để Ultralytics YOLO tự động liên kết nhãn mà không gây lỗi.

In [ ]:
# ==========================================================
# BƯỚC 2: XÁC ĐỊNH ĐƯỜNG DẪN & THỐNG KÊ CHI TIẾT 3 TẬP
# ==========================================================

CURRENT_DIR = Path(".").resolve()
if (CURRENT_DIR / "Thermal").exists():
    DATASET_ROOT = (CURRENT_DIR / "Thermal").resolve()
elif (CURRENT_DIR.parent / "Thermal").exists():
    DATASET_ROOT = (CURRENT_DIR.parent / "Thermal").resolve()
else:
    DATASET_ROOT = Path(r"D:\Đồ án Fall 26\Data aggregation\Thermal").resolve()

print(f"📁 Thư mục dataset gốc: {DATASET_ROOT}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_split(split_name):
    split_dir = DATASET_ROOT / split_name
    # Kiểm tra cả 'image' và 'images'
    img_dir = split_dir / "image" if (split_dir / "image").exists() else split_dir / "images"
    lbl_dir = split_dir / "label" if (split_dir / "label").exists() else split_dir / "labels"
    
    imgs = [f for f in img_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS] if img_dir.exists() else []
    lbls = [f for f in lbl_dir.iterdir() if f.suffix.lower() == ".txt" and f.stem != "classes"] if lbl_dir.exists() else []
    
    img_stems = {f.stem for f in imgs}
    lbl_stems = {f.stem for f in lbls}
    matched = len(img_stems & lbl_stems)
    return len(imgs), len(lbls), matched

tr_i, tr_l, tr_m = count_split("Train")
va_i, va_l, va_m = count_split("Val")
te_i, te_l, te_m = count_split("Test")

print("\n" + "=" * 72)
print(f"{'TẬP DỮ LIỆU':<15} | {'SỐ LƯỢNG ẢNH':<15} | {'SỐ NHÃN TXT':<15} | {'ĐỘ KHỚP':<12}")
print("=" * 72)
print(f"{'Train (Huấn luyện)':<15} | {tr_i:<15,} | {tr_l:<15,} | {'✅ 100% Khớp' if tr_i == tr_m else '❌ Lệch'}")
print(f"{'Val (Kiểm định)':<15} | {va_i:<15,} | {va_l:<15,} | {'✅ 100% Khớp' if va_i == va_m else '❌ Lệch'}")
print(f"{'Test (Đánh giá)':<15} | {te_i:<15,} | {te_l:<15,} | {'✅ 100% Khớp' if te_i == te_m else '❌ Lệch'}")
print("=" * 72)
print(f"Tổng cộng dataset: {tr_i + va_i + te_i:,} ảnh và {tr_l + va_l + te_l:,} nhãn txt.")

## 🔍 Bước 3: Trực Quan Hóa Mẫu Ảnh Nhiệt & Ground Truth Bounding Box

Chúng ta sẽ hiển thị ngẫu nhiên các ảnh trong tập dữ liệu cùng các bounding box thật được giải mã từ file nhãn `.txt` YOLO ($x_{center}, y_{center}, w, h$) để kiểm chứng trực quan.

In [ ]:
# ==========================================================
# BƯỚC 3: HIỂN THỊ MẪU ẢNH NHIỆT VÀ BOUNDING BOX THỰC TẾ
# ==========================================================
import random

train_img_dir = DATASET_ROOT / "Train" / "images"
train_lbl_dir = DATASET_ROOT / "Train" / "labels"

def plot_sample_with_boxes(img_path, lbl_path, class_names={0: "person"}):
    img = cv2.imread(str(img_path))
    if img is None:
        return None, 0
    h, w, _ = img.shape
    
    box_count = 0
    if lbl_path.exists():
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    
                    # Chuyển đổi tọa độ YOLO sang Pixel Bounding Box
                    x1 = int((xc - bw / 2) * w)
                    y1 = int((yc - bh / 2) * h)
                    x2 = int((xc + bw / 2) * w)
                    y2 = int((yc + bh / 2) * h)
                    
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label_name = class_names.get(cls_id, f"id_{cls_id}")
                    cv2.putText(img, label_name, (x1, max(y1 - 5, 12)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
                    box_count += 1
                    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb, box_count

# Lấy ngẫu nhiên 4 ảnh từ tập Train
all_train_imgs = list(train_img_dir.glob("*.jpg"))
sample_imgs = random.sample(all_train_imgs, min(4, len(all_train_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, img_p in enumerate(sample_imgs):
    lbl_p = train_lbl_dir / f"{img_p.stem}.txt"
    annotated_img, count = plot_sample_with_boxes(img_p, lbl_p)
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(f"{img_p.name} ({count} person)", fontsize=11)
    axes[idx].axis("off")

plt.suptitle("Mẫu Ảnh Nhiệt Hồng Ngoại Kèm Ground Truth Bounding Box", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## ⚙️ Bước 4: Tạo File Cấu Hình Huấn Luyện `data.yaml`

File `data.yaml` chỉ định 3 tập riêng biệt:
- `train`: `Train/images` (19,802 ảnh)
- `val`: `Val/images` (2,200 ảnh)
- `test`: `Test/images` (5,548 ảnh)
- `names`: `0: person`

In [ ]:
# ==========================================================
# BƯỚC 4: TẠO FILE CẤU HÌNH DATA.YAML
# ==========================================================

dataset_path_str = DATASET_ROOT.as_posix()

data_yaml_dict = {
    "path": dataset_path_str,
    "train": "Train/images",
    # Tu dong dung Test lam Val neu chua tach thu muc Val
    "val": "Val/images" if (DATASET_ROOT / "Val").exists() else "Test/images",
    "test": "Test/images",
    "names": {
        0: "person"
    }
}

yaml_path = DATASET_ROOT.parent / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_dict, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"✅ Đã tạo file data.yaml tại: {yaml_path.resolve()}\n")
print("--- Nội dung file data.yaml ---")
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

## 🧠 Bước 5: Khởi Tạo Mô Hình YOLO11s (`yolo11s.pt`)

Mô hình **YOLO11s** (Small) sở hữu cấu trúc mạng cải tiến từ Ultralytics (C3k2 blocks, SPPF, C2PSA attention) mang lại:
- Khả năng trích xuất đặc trưng hình thái người trong điều kiện ánh sáng yếu/ảnh nhiệt vượt trội.
- Cân bằng tối ưu giữa mAP và FPS thời gian thực khi chạy trên camera UAV/drone hoặc máy bay trinh sát.

In [ ]:
# ==========================================================
# BƯỚC 5: LOAD PRE-TRAINED WEIGHTS YOLO11s
# ==========================================================
from ultralytics import YOLO

MODEL_NAME = "yolo11s.pt"
print(f"📥 Đang khởi tạo mô hình {MODEL_NAME}...")

model = YOLO(MODEL_NAME)

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f"✅ Tải mô hình thành công: {MODEL_NAME}")
print(f"📊 Tổng số tham số (Parameters)        : {total_params:,}")
print(f"🔥 Số tham số huấn luyện (Trainable): {trainable_params:,}")

## 🏋️‍♂️ Bước 6: Cấu Hình Siêu Tham Số & Tiến Hành Huấn Luyện (Training)

### 💡 Cấu hình chi tiết:
| Tham số | Giá trị gợi ý | Ý nghĩa |
| :--- | :---: | :--- |
| `epochs` | `50` | Số vòng lặp huấn luyện (với 19.8k ảnh, 50 epochs là rất lớn và đủ hội tụ cao) |
| `batch` | `16` | Kích thước batch phù hợp VRAM 8GB (RTX 4060). Nếu đầy VRAM hãy chỉnh về `8` |
| `imgsz` | `640` | Kích thước chuẩn cân bằng độ phân giải và tốc độ |
| `device` | `0` | GPU NVIDIA (CUDA) |
| `patience` | `15` | Dừng sớm (Early stopping) nếu sau 15 epochs không cải thiện mAP trên tập Val |
| `save_period` | `10` | Tự động lưu checkpoint mỗi 10 epochs |
| `amp` | `True` | Kích hoạt Automatic Mixed Precision (FP16) |
| `mosaic` / `mixup` | `1.0` / `0.1` | Tăng cường ảnh ghép giúp nhận diện đối tượng che khuất hoặc xa gần |

In [ ]:
# ==========================================================
# BƯỚC 6: THIẾT LẬP THAM SỐ & TIẾN HÀNH HUẤN LUYỆN
# ==========================================================

EPOCHS = 50           # Số epoch huấn luyện (30 - 100)
BATCH_SIZE = 16       # Kích thước batch (giảm xuống 8 nếu báo lỗi CUDA Out of Memory)
IMAGE_SIZE = 640      # Kích thước ảnh đầu vào (640x640)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PROJECT_DIR = "runs/train"
EXPERIMENT_NAME = "yolo11s_thermal_person"

print(f"🚀 KHỞI ĐỘNG HUẤN LUYỆN YOLO11s:")
print(f"   - Dataset file : {yaml_path.resolve()}")
print(f"   - Số Epochs    : {EPOCHS}")
print(f"   - Batch Size   : {BATCH_SIZE}")
print(f"   - Kích thước ảnh: {IMAGE_SIZE}")
print(f"   - Thiết bị     : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print("=" * 60)

train_results = model.train(
    data=str(yaml_path.resolve()),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    workers=4,
    project=PROJECT_DIR,
    name=EXPERIMENT_NAME,
    patience=15,             # Early stopping dựa trên tập Val
    save=True,
    save_period=10,
    val=True,                # Đánh giá sau mỗi epoch trên tập Val
    plots=True,
    amp=True,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    exist_ok=True
)

best_weights = Path(PROJECT_DIR) / EXPERIMENT_NAME / "weights" / "best.pt"
last_weights = Path(PROJECT_DIR) / EXPERIMENT_NAME / "weights" / "last.pt"

print("\n🎉 CHÚC MỪNG: QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT!")
print(f"🏆 Trọng số tốt nhất (Best weights): {best_weights.resolve()}")
print(f"💾 Trọng số cuối cùng (Last weights): {last_weights.resolve()}")

## 📊 Bước 7: Đánh Giá Chi Tiết Trên Cả Tập Val & Tập Test Độc Lập

Nạp trọng số tốt nhất (`best.pt`) để thực hiện đánh giá toàn diện trên cả tập **Val (2,200 ảnh)** và tập **Test (5,548 ảnh)**:
- **Precision (Độ chuẩn xác)**
- **Recall (Độ phủ / Tỉ lệ bắt được người)**
- **mAP@50 (Mean Average Precision tại IoU=0.5)**
- **mAP@50-95 (mAP trung bình trên dải IoU 0.5:0.95)**

In [ ]:
# ==========================================================
# BƯỚC 7: ĐÁNH GIÁ ĐỘ CHÍNH XÁC MÔ HÌNH TRÊN TẬP VAL VÀ TẬP TEST
# ==========================================================

trained_model = YOLO(str(best_weights))

# 1. Đánh giá trên tập Val
print("📊 Đang đánh giá trên tập Val...")
val_res = trained_model.val(data=str(yaml_path.resolve()), split="val", imgsz=IMAGE_SIZE, device=DEVICE)

# 2. Đánh giá trên tập Test độc lập
print("\n📊 Đang đánh giá trên tập Test độc lập...")
test_res = trained_model.val(data=str(yaml_path.resolve()), split="test", imgsz=IMAGE_SIZE, device=DEVICE)

print("\n" + "=" * 65)
print(f"{'CHỈ SỐ ĐÁNH GIÁ':<20} | {'TẬP VAL (2,200 ảnh)':<20} | {'TẬP TEST (5,548 ảnh)':<20}")
print("=" * 65)
print(f"{'Precision (P)':<20} | {val_res.box.mp * 100:.2f}%{'':<14} | {test_res.box.mp * 100:.2f}%")
print(f"{'Recall (R)':<20} | {val_res.box.mr * 100:.2f}%{'':<14} | {test_res.box.mr * 100:.2f}%")
print(f"{'mAP@50':<20} | {val_res.box.map50 * 100:.2f}%{'':<14} | {test_res.box.map50 * 100:.2f}%")
print(f"{'mAP@50-95':<20} | {val_res.box.map * 100:.2f}%{'':<14} | {test_res.box.map * 100:.2f}%")
print("=" * 65)

## 📈 Bước 8: Trực Quan Hóa Đồ Thị Huấn Luyện (Training Curves)

Hiển thị trực tiếp các đồ thị do Ultralytics sinh ra trong thư mục `runs/train/yolo11s_thermal_person/`:
1. **results.png**: Đồ thị diễn tiến loss (Box, Class, DFL) và mAP qua từng epoch.
2. **confusion_matrix.png**: Ma trận nhầm lẫn.
3. **PR_curve.png**: Đường cong Precision - Recall.
4. **F1_curve.png**: Đường cong F1 theo ngưỡng tin cậy (Confidence).

In [ ]:
# ==========================================================
# BƯỚC 8: HIỂN THỊ ĐỒ THỊ HUẤN LUYỆN VÀ ĐÁNH GIÁ
# ==========================================================
exp_folder = Path(PROJECT_DIR) / EXPERIMENT_NAME

chart_files = [
    (exp_folder / "results.png", "Biểu đồ Tiến trình Huấn luyện (Loss & mAP)"),
    (exp_folder / "confusion_matrix.png", "Ma trận nhầm lẫn (Confusion Matrix)"),
    (exp_folder / "PR_curve.png", "Đường cong Precision - Recall (PR Curve)"),
    (exp_folder / "F1_curve.png", "Đường cong F1-Score theo Confidence")
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for i, (f_path, title) in enumerate(chart_files):
    ax = axes[i // 2, i % 2]
    if f_path.exists():
        img = Image.open(f_path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"Chưa có file:\n{f_path.name}", ha="center", va="center", fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 🔮 Bước 9: Chạy Dự Đoán Thử Nghiệm Trên Ảnh Mới (Inference Demo)

Lấy ngẫu nhiên các bức ảnh từ tập `Test` mà mô hình chưa từng được tối ưu gradients trực tiếp, chạy nhận diện với ngưỡng tin cậy `conf=0.30` và vẽ bounding box dự đoán.

In [ ]:
# ==========================================================
# BƯỚC 9: INFERENCE TRỰC TIẾP TRÊN ẢNH TẬP TEST
# ==========================================================
test_img_dir = DATASET_ROOT / "Test" / "images"
all_test_imgs = list(test_img_dir.glob("*.jpg"))
demo_imgs = random.sample(all_test_imgs, min(6, len(all_test_imgs)))

print(f"🔮 Đang chạy dự đoán trên {len(demo_imgs)} ảnh test ngẫu nhiên...")

predictions = trained_model.predict(
    source=[str(p) for p in demo_imgs],
    conf=0.30,        # Ngưỡng tin cậy confidence threshold
    iou=0.45,         # Ngưỡng lọc trùng NMS IoU threshold
    device=DEVICE
)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, res in enumerate(predictions):
    res_bgr = res.plot()
    res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
    detected_count = len(res.boxes)
    
    axes[idx].imshow(res_rgb)
    axes[idx].set_title(f"{demo_imgs[idx].name}\nPhát hiện: {detected_count} người", fontsize=11)
    axes[idx].axis("off")

plt.suptitle("Kết Quả Phát Hiện Người Bằng YOLO11s Trên Ảnh Nhiệt (Tập Test)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 📦 Bước 10: Xuất Mô Hình Để Triển Khai Thực Tế (ONNX / TensorRT Export)

Sau khi huấn luyện thành công, mô hình có thể được xuất sang định dạng **ONNX** để nạp vào hệ thống máy tính nhúng trên thiết bị bay UAV drone (như NVIDIA Jetson Orin / Xavier / Nano) hoặc tích hợp vào phần mềm cứu hộ cứu nạn thời gian thực bằng C++ / Python.

In [ ]:
# ==========================================================
# BƯỚC 10: XUẤT MÔ HÌNH SANG ĐỊNH DẠNG ONNX
# ==========================================================
print("📦 Đang xuất mô hình sang ONNX...")

onnx_output = trained_model.export(
    format="onnx",
    dynamic=True,
    simplify=True,
    opset=12
)

print(f"\n✅ Đã xuất mô hình ONNX thành công tại: {onnx_output}")
print("💡 Mô hình đã sẵn sàng để tích hợp vào camera hồng ngoại / UAV cứu nạn!")